In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE = Path("../data/raw")

meta_data_df = pd.read_csv(BASE / "all_metadata.csv")

EXPECTED_ROWS = 602
EXPECTED_COLS_BASELINE = 140 
EXPECTED_COLS_ATTACK = 141


### 1- Scan each CSV file individually and list the ones that have mismatched (inconsistent) shapes or missing values.

In [3]:
# First, we will go through all baseline and attack CSV files listed in the metadata one by one,
# and list the ones that have row/column mismatches or contain missing (null) values.

# List to store problematic CSV files:
problematic_csvs = []

for _, row in meta_data_df.iterrows():
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path)

    if row["Scenario"] == "baseline":
        expected_cols = EXPECTED_COLS_BASELINE
    else:
        expected_cols = EXPECTED_COLS_ATTACK

    # Checks
    row_ok = df.shape[0] == EXPECTED_ROWS 
    col_ok = df.shape[1] == expected_cols

    #Check whether there are any missing values:
    missing_count = df.isnull().sum().sum() # the first sum computes null counts per column, the second sum gives the total

    if not row_ok or not col_ok or missing_count > 0:
        problematic_csvs.append({
            "data_file"  : row["data_file"],
            "Scenario"   : row["Scenario"],
            "actual_rows": df.shape[0],
            "actual_cols": df.shape[1],
            "expected_rows" : EXPECTED_ROWS,
            "expected_cols" : expected_cols,
            "missing_values" : missing_count

        })


In [4]:
problematic_df = pd.DataFrame(problematic_csvs)

print(f"Total CSVs scanned: : {len(meta_data_df)}")
print(f"Number of problematic CSVs: {len(problematic_df)}")

# pd.set_option("display.max_rows", None) --> --> uncomment this line to inspect all 194 mismatched CSVs.
display(problematic_df)

Total CSVs scanned: : 1920
Number of problematic CSVs: 194


,data_file,Scenario,actual_rows,actual_cols,expected_rows,expected_cols,missing_values
0,./data/raw/baseline/fsw_data_4_180.0_0.csv,baseline,602,140,602,140,4214
1,./data/raw/baseline/fsw_data_5_72.0_0.csv,baseline,602,140,602,140,4214
2,./data/raw/baseline/fsw_data_5_180.0_0.csv,baseline,602,140,602,140,4214
3,./data/raw/baseline/fsw_data_5_252.0_0.csv,baseline,602,140,602,140,4214
4,./data/raw/baseline/fsw_data_6_36.0_0.csv,baseline,602,140,602,140,4214
...,...,...,...,...,...,...,...
189,./data/raw/attack_rwc/fsw_data_9_288.0_-50.csv,attack_rwc,602,141,602,141,4214
190,./data/raw/attack_rwc/fsw_data_9_288.0_-25.csv,attack_rwc,602,141,602,141,4214
191,./data/raw/attack_rwc/fsw_data_9_288.0_0.csv,attack_rwc,602,141,602,141,4214
192,./data/raw/attack_rwc/fsw_data_9_288.0_25.csv,attack_rwc,602,141,602,141,4214


In [5]:
# In the metadata EDA, we had obtained the following FSW_rows output:
# 602    1918
# 510       1
# 341       1
# Confirming this output, these 2 CSV files with missing rows appear at index 188 and 132!

# Also, in the metadata EDA, 192 files had missing columns (7 missing columns each);
# here we observe the same thing: 4214/612 = 7, meaning 7 columns are missing per row.
# Let's check which columns those are:
# We inspect a sample CSV file that contains missing values, to see which columns are missing:

df = pd.read_csv("../data/raw/baseline/fsw_data_4_180.0_0.csv")

# Which columns have NaN values?
empty_columns = df.columns[df.isnull().any()].tolist()
print(empty_columns)

print(df[empty_columns].isnull().sum())


['RawPrimaryGenericPointData[0]()', 'RawPrimaryGenericPointData[1]()', 'RawPrimaryGenericPointData[2]()', 'RawSecondaryGenericPointData[0]()', 'RawSecondaryGenericPointData[1]()', 'RawSecondaryGenericPointData[2]()', 'AngleToPrimaryTarget(rad)']
RawPrimaryGenericPointData[0]()      602
RawPrimaryGenericPointData[1]()      602
RawPrimaryGenericPointData[2]()      602
RawSecondaryGenericPointData[0]()    602
RawSecondaryGenericPointData[1]()    602
RawSecondaryGenericPointData[2]()    602
AngleToPrimaryTarget(rad)            602
dtype: int64


In [6]:
# In the previous cell, we inspected which columns contained missing values. Since these columns do not belong
# to the features we already selected, we re-checked the entire metadata this time based on rows, columns,
# and specifically whether the columns belonging to our selected features have any missing values.
# In other words, those missing values do not matter to us since they belong to features we haven't selected.

SELECTED_FEATURES = [
    "Q_B_I[0](1)", "Q_B_I[1](1)", "Q_B_I[2](1)", "Q_B_I[3](1)", 
"AttitudeError[0](rad)", "AttitudeError[1](rad)", "AttitudeError[2](rad)",
"AngVel_B_I[0](rad/sec)", "AngVel_B_I[1](rad/sec)", "AngVel_B_I[2](rad/sec)", "AngVelMag_B_I(rad/sec)",
"SensedWheelSpeed__RWA_A(rad/sec)", "SensedWheelSpeed__RWA_B(rad/sec)", "SensedWheelSpeed__RWA_C(rad/sec)",
"WheelCmd__RWA_A(N*m)", "WheelCmd__RWA_B(N*m)", "WheelCmd__RWA_C(N*m)",
"DesiredWheelCommand[0](N*m)", "DesiredWheelCommand[1](N*m)", "DesiredWheelCommand[2](N*m)",
"TotalTorqueRodCommand[0](A*m^2)", "TotalTorqueRodCommand[1](A*m^2)", "TotalTorqueRodCommand[2](A*m^2)",
"BField_B__TAM[0](T)", "BField_B__TAM[1](T)", "BField_B__TAM[2](T)"
]

# This time we scan again to check whether our selected features have missing values.
# If not, even if the total column count is missing compared to the expected count, we will NOT
# exclude that CSV file, since it still contains all of our selected features!

problematic_csvs_final = []

for _, row in meta_data_df.iterrows():
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path)

    if row["Scenario"] == "baseline":
        expected_cols = EXPECTED_COLS_BASELINE
    else:
        expected_cols = EXPECTED_COLS_ATTACK


    row_ok = df.shape[0] == EXPECTED_ROWS
    col_ok = df.shape[1] == expected_cols

    # We only check for missing values within the selected features
    missing_in_features = df[SELECTED_FEATURES].isnull().sum().sum()

    if not row_ok or not col_ok or missing_in_features > 0:
        problematic_csvs_final.append({
            "data_file"           : row["data_file"],
            "Scenario"            : row["Scenario"],
            "actual_rows"         : df.shape[0],
            "expected_rows"       : EXPECTED_ROWS,
            "actual_cols"         : df.shape[1],
            "expected_cols"       : expected_cols,
            "missing_in_features" : missing_in_features
        })

In [7]:
problematic_df_final = pd.DataFrame(problematic_csvs_final)

print(f"Total CSVs scanned: {len(meta_data_df)}")
print(f"Number of problematic CSVs: {len(problematic_df_final)}")

display(problematic_df_final)

Total CSVs scanned: 1920
Number of problematic CSVs: 2


,data_file,Scenario,actual_rows,expected_rows,actual_cols,expected_cols,missing_in_features
0,./data/raw/attack_rwb/fsw_data_12_144.0_-50.csv,attack_rwb,510,602,141,141,1530
1,./data/raw/attack_rwc/fsw_data_9_108.0_-50.csv,attack_rwc,341,602,141,141,1023


### 2- Filter the metadata: Drop the problematic CSVs from the metadata


In [8]:
# Only 2 CSV files turned out to contain missing values in our selected features;
# we will exclude these to create a clean metadata.

files_to_drop = problematic_df_final["data_file"].tolist()

is_problematic = meta_data_df["data_file"].isin(files_to_drop)
clean_metadata_df = meta_data_df[~is_problematic]

print(f"Original metadata: {len(meta_data_df)} rows")
print(f"Cleaned metadata: {len(clean_metadata_df)} rows")
print(f"Dropped: {len(meta_data_df) - len(clean_metadata_df)} rows")

Original metadata: 1920 rows
Cleaned metadata: 1918 rows
Dropped: 2 rows


In [9]:
# Save clean_metadata
import os
os.makedirs("../data/processed", exist_ok = True)

clean_metadata_df.to_csv("../data/processed/clean_metadata.csv", index = False)

### 3- Class Balance After Dropping

In [10]:
print("Scenario-Wise Class Distribution (File Level):")
print(clean_metadata_df["Scenario"].value_counts())

Scenario-Wise Class Distribution (File Level):
Scenario
attack_rwa    600
attack_rwb    599
attack_rwc    599
baseline      120
Name: count, dtype: int64


In [11]:
all_dfs = []

for _, row in clean_metadata_df.iterrows():
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path, usecols=["result_label"])
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index = True)

print("Scenario-Wise Class Distribution (Row Level):")
print(combined_df["result_label"].value_counts())

Scenario-Wise Class Distribution (Row Level):
result_label
baseline    615236
RWA         180000
RWB         179700
RWC         179700
Name: count, dtype: int64


### 4- Process & save each clean CSV:

In [12]:
processed_paths = []

for _, row in clean_metadata_df.iterrows():
    # 1- Open the CSV
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path)

    # 2- Keep only the selected 26 features + result_label columns
    df_processed = df[SELECTED_FEATURES + ["result_label"]]

    # 3- Add label_binary: baseline → 0, attack → 1
    df_processed = df_processed.copy() 
    df_processed["label_binary"] = (df_processed["result_label"] != "baseline").astype(int)

    # 4- Save: preserve the original file name and scenario folder structure
    file_name = Path(row["data_file"]).name # extracts only the file name from the long path, e.g. fsw_data_1_0.0_0.csv 
    scenario = row["Scenario"]
    save_path = Path("../data/processed") / scenario / file_name

    df_processed.to_csv(save_path, index= False)
    processed_paths.append(str(save_path))

print(f"{len(processed_paths)} CSV files were processed and saved.")


1918 CSV files were processed and saved.


In [13]:
# Add the data_file_processed column to clean_metadata_df
clean_metadata_df = clean_metadata_df.copy()
clean_metadata_df["data_file_processed"] = processed_paths

# Update clean_metadata.csv with the data_file_processed column included
clean_metadata_df.to_csv("../data/processed/clean_metadata.csv", index=False)